# Homography estimation and panorama stitching

*graphics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/graphics/homography_stitching)

Two photos of a flat poster are related by a $3\times3$ homography. Each feature match gives two linear equations. The normalised direct linear transform (DLT) solves them with an SVD, and RANSAC fits thousands of 4-match hypotheses as one batched SVD to throw out the wrong matches. The second view is then warped and feather-blended with the first into a panorama.

**Vector form**

$$\lambda\,\mathbf x' = H\,\mathbf x \;\Leftrightarrow\; \mathbf x' \times H\mathbf x = \mathbf 0 \qquad \hat{\mathbf h} = \arg\min_{\lVert\mathbf h\rVert = 1} \lVert A\,\mathbf h\rVert \qquad H^\star = \arg\max_{H_b} \#\{n : \lVert \mathbf x'_n - H_b \mathbf x_n\rVert < \tau\}$$

**Index (tensor) form**

$$x'_{ni} \propto H_{ij}\,x_{nj} \qquad A_{b,(nr),a}\,h_{ba} = 0,\ \ A = U\,\Sigma\,V^\top \Rightarrow h_{ba} = V_{b,a9} \qquad e_{bn} = \Big\lVert x'_{n} - \frac{H_{bij}\,x_{nj}}{H_{b3j}\,x_{nj}} \Big\rVert \qquad b^\star = \arg\max_b \sum_n [e_{bn} < \tau]$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{ni},\ x'_{ni}$ | `tensors.matches(Hs, ...)` | (N, 2) | matched pixel positions in view 1 and view 2 (known poster points + noise, some replaced by random wrong matches) |
| $H_{ij}\,x_{nj}$ | `einsum('bij,nj->bni', H, to_homogeneous(x))` | (B, N, 3) | repeated $j$ ⇒ summed, then divide by the third component (the projective scale) |
| $T,\ T'$ | `normalisation(x)` | (B, 3, 3) | shift to the centroid and scale to mean distance $\sqrt 2$ (Hartley normalisation) |
| $A_{b,(nr),a}$ | `dlt_matrix(n1, n2)` | (B, 2N, 9) | two rows per match from $\mathbf x' \times H\mathbf x = 0$, linear in the nine entries $h_a$ |
| $h_{ba} = V_{b,a9}$ | `torch.linalg.svd(A)[2][:, -1]` | (B, 9) | right singular vector of the smallest singular value; then $H = T'^{-1}\hat H\,T$ |
| $b$ | `rand(B, N).argsort(1)[:, :4]` | (B, 4) | RANSAC hypotheses: $B$ random 4-match samples, all solved by one batched SVD of shape $(B, 8, 9)$ |
| $e_{bn}$ | `transfer_error(H_all, x1, x2)` | (B, N) | reprojection error of every match under every hypothesis |
| $b^\star$ | `(err < tau).sum(1).argmax()` | () | the hypothesis with the largest consensus set; refit by DLT on its inliers |
| $I(H\,\mathbf p)$ | `grid_sample(img2, H12 p)` | (h, w, 3) | inverse warp: each panorama pixel looks up where it lands in view 2; blended with feathering weights |

A camera looking at a plane maps plane points to pixels by $H = K\,[\mathbf r_1\ \mathbf r_2\ \mathbf t]$, so two views of the same plane are related by $H_2 H_1^{-1}$: eight numbers, because the scale is free. Each match fixes two of them, so four matches in general position determine $H$. More matches are combined in the least squares sense by the DLT. Without Hartley's normalisation the entries of $A$ range from 1 to $10^4$ (pixel squared) and the SVD loses accuracy. Least squares is ruined by wrong matches: a few outliers drag the naive panorama visibly out of alignment. RANSAC (Fischler & Bolles, 1981) instead fits $H$ to many random minimal samples and keeps the one most matches agree with. With an outlier fraction $\varepsilon$, a 4-match sample is clean with probability $(1-\varepsilon)^4$, so a few dozen samples are enough in theory. Here thousands are solved at once because batched SVDs are cheap. The remaining error on the inliers matches the pixel noise. The method follows Hartley & Zisserman, Multiple View Geometry in Computer Vision. Real pipelines get the matches from feature detectors such as SIFT or ORB.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/graphics/homography_stitching/main.py
"""Main loop: parameters → two views and noisy matches → DLT and batched RANSAC → panorama → plot."""

import math
import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import dlt, transfer_error

SIZE = (192, 144)  # image width, height in pixels
FOCAL = 170.0


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n_hyp = 200 if fast_mode() else params["hypotheses"]
    Hs, imgs, H_true = tensors.views(SIZE, FOCAL, dev)
    x1, x2, is_out = tensors.matches(Hs, SIZE, params["matches"], params["noise"], params["outliers"], seed=0)
    eps = float(is_out.float().mean())
    print(f"two {SIZE[0]}×{SIZE[1]} views of a planar poster; {len(x1)} matches, {int(is_out.sum())} of them wrong")
    print("true view-1 → view-2 homography (H₃₃ = 1):")
    for row in H_true.tolist():
        print("   " + "  ".join(f"{v:9.4f}" for v in row))

    H_all = dlt(x1[None], x2[None])[0]  # least squares on every match, outliers included
    p_good = (1 - eps) ** 4
    need = math.log(0.001) / math.log(1 - p_good) if p_good < 1 else 1
    print(f"a random 4-match sample is outlier-free with probability {p_good:.3f}; {need:.0f} samples give 99.9% odds")
    start = time.perf_counter()
    H, inliers, counts = solve.ransac(x1, x2, n_hyp, params["threshold"], torch.Generator().manual_seed(1))
    elapsed = time.perf_counter() - start
    print(f"RANSAC: {n_hyp} hypotheses as one batched SVD of shape ({n_hyp}, 8, 9) in {1000 * elapsed:.0f} ms")
    print(f"best hypothesis has {int(counts.max())} inliers; median hypothesis {int(counts.median())}")
    tp = int((inliers & ~is_out).sum())
    print(
        f"inliers kept {inliers.sum().item()}: {tp} true matches, {int((inliers & is_out).sum())} wrong ones "
        f"(recall {tp / int((~is_out).sum()):.2f})"
    )

    e_ransac = transfer_error(H[None], x1, x2)[0]
    e_all = transfer_error(H_all[None], x1, x2)[0]
    rms = float(e_ransac[inliers].pow(2).mean().sqrt())
    print(
        f"RMS reprojection error on the inliers: {rms:.2f} px (noise σ = {params['noise']} px per axis in both "
        f"images, so ≈ 2σ = {2 * params['noise']:.2f} px expected)"
    )
    ce_r, ce_a = solve.corner_error(H, H_true, SIZE), solve.corner_error(H_all, H_true, SIZE)
    print(f"corner error vs the true H: RANSAC + DLT {ce_r:.2f} px, plain DLT on all matches {ce_a:.1f} px")

    pano_r, origin = solve.stitch(imgs, H)
    pano_a, origin_a = solve.stitch(imgs, H_all)
    print(f"panorama: {pano_r.shape[1]}×{pano_r.shape[0]} pixels from two {SIZE[0]}×{SIZE[1]} views")
    return plot.figure(
        imgs,
        x1,
        x2,
        is_out,
        inliers,
        pano_r,
        pano_a,
        e_ransac,
        e_all,
        params["threshold"],
        (ce_r, ce_a),
        out_dir / "homography_stitching.png",
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/graphics/homography_stitching/equation.py
r"""Projective geometry → tensor translation.

Two photos of a plane are related by a homography, a 3 × 3 matrix acting on homogeneous pixel coordinates:

    x'_i ∝ H_{ij} x_j        (j summed; x = (u, v, 1), equality up to the scale x'_3).

Each match x ↔ x' gives two linear equations in the nine entries h_a of H, from x' × (H x) = 0:

    A_{n0a} h_a = 0,   A_{n0} = (−u, −v, −1, 0, 0, 0, u'u, u'v, u')
    A_{n1a} h_a = 0,   A_{n1} = (0, 0, 0, −u, −v, −1, v'u, v'v, v')

The direct linear transform (DLT) takes h as the unit vector minimising |A h|: the right singular vector of A with
the smallest singular value. Hartley's normalisation (shift each point set to its centroid and scale it to mean
distance √2) makes the columns of A comparable and the SVD well conditioned; H = T'^{-1} Ĥ T afterwards.

Everything here takes a leading batch index b, so thousands of 4-point RANSAC hypotheses are solved by one batched
SVD of shape (B, 8, 9).
"""

import torch


def to_homogeneous(x: torch.Tensor) -> torch.Tensor:
    """(..., 2) → (..., 3) by appending 1."""
    return torch.cat([x, torch.ones_like(x[..., :1])], -1)


def apply_homography(H: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    r"""x'_{bni} = H_{bij} x_{bnj} / (H_{b3j} x_{bnj})   (j summed).   H (B, 3, 3), x (B, N, 2) or (N, 2) → (B, N, 2)"""
    xh = to_homogeneous(x)
    eq = "bij,bnj->bni" if xh.dim() == 3 else "bij,nj->bni"
    y = torch.einsum(eq, H, xh)
    w = y[..., 2:]
    return y[..., :2] / torch.where(w.abs() < 1e-12, torch.full_like(w, 1e-12), w)


def normalisation(x: torch.Tensor) -> torch.Tensor:
    """Similarity T (B, 3, 3) moving each point set x (B, N, 2) to centroid 0 and mean distance √2."""
    centre = x.mean(dim=1)  # (B, 2)
    scale = 2**0.5 / (x - centre[:, None]).norm(dim=-1).mean(dim=1).clamp(min=1e-12)  # (B,)
    T = torch.zeros(x.shape[0], 3, 3, dtype=x.dtype, device=x.device)
    T[:, 0, 0] = T[:, 1, 1] = scale
    T[:, :2, 2] = -scale[:, None] * centre
    T[:, 2, 2] = 1
    return T


def dlt_matrix(x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
    """A_{b,(n r),a}: two rows per match. x1, x2 (B, N, 2) → (B, 2N, 9)."""
    u, v = x1[..., 0], x1[..., 1]
    u2, v2 = x2[..., 0], x2[..., 1]
    one, zero = torch.ones_like(u), torch.zeros_like(u)
    row_u = torch.stack([-u, -v, -one, zero, zero, zero, u2 * u, u2 * v, u2], -1)  # (B, N, 9)
    row_v = torch.stack([zero, zero, zero, -u, -v, -one, v2 * u, v2 * v, v2], -1)
    return torch.stack([row_u, row_v], dim=2).reshape(x1.shape[0], -1, 9)


def dlt(x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
    r"""Normalised DLT for a batch of point sets: h = argmin_{|h|=1} |A h| by SVD.   (B, N, 2) × 2 → H (B, 3, 3)

    H is scaled so that H_33 = 1 (a homography is only defined up to scale)."""
    T1, T2 = normalisation(x1), normalisation(x2)
    n1 = apply_homography(T1, x1)  # (B, N, 2) normalised points
    n2 = apply_homography(T2, x2)
    A = dlt_matrix(n1, n2)
    _, _, Vh = torch.linalg.svd(A)  # batched SVD; full_matrices gives Vh (B, 9, 9) even for 4 points (8 rows)
    H_hat = Vh[:, -1].reshape(-1, 3, 3)  # right singular vector of the smallest singular value
    H = torch.linalg.solve(T2, H_hat @ T1)  # undo the normalisation: H = T2⁻¹ Ĥ T1
    h33 = H[:, 2:, 2:]
    return H / torch.where(h33.abs() < 1e-12, torch.full_like(h33, 1e-12), h33)


def transfer_error(H: torch.Tensor, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
    r"""e_{bn} = |x2_n − H_b x1_n| in pixels for every hypothesis b and match n.   H (B, 3, 3), x (N, 2) → (B, N)"""
    err = (apply_homography(H, x1) - x2).norm(dim=-1)
    return torch.nan_to_num(err, nan=1e9, posinf=1e9)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/graphics/homography_stitching/tensors.py
"""Build the concrete tensors: a procedural poster on the plane z = 0, two cameras that see it, the two rendered
views and noisy feature matches with outliers.

Pixel convention: pixel (column i, row j) has its centre at image coordinates (u, v) = (i, j).
"""

import math

import torch

POSTER = (2.4, 1.5)  # half width and half height of the poster in world units
BACKGROUND = 0.94


def texture(xy: torch.Tensor) -> torch.Tensor:
    """Colour of the poster at plane points (..., 2) → (..., 3): smooth colour waves, a grid and some discs."""
    x, y = xy[..., 0], xy[..., 1]
    rgb = torch.stack(
        [
            0.55 + 0.35 * torch.sin(2.1 * x + 0.5) * torch.cos(1.7 * y),
            0.55 + 0.35 * torch.sin(1.3 * y - 0.8 * x + 1.0),
            0.60 + 0.30 * torch.cos(1.1 * x * y + 0.3 * x),
        ],
        -1,
    )
    lines = ((torch.frac(2 * x + 100) < 0.05) | (torch.frac(2 * y + 100) < 0.05))[..., None]
    rgb = torch.where(lines, 0.25 * rgb, rgb)
    gen = torch.Generator().manual_seed(7)
    centres = (torch.rand(14, 2, generator=gen) * 2 - 1) * torch.tensor([2.1, 1.25])
    radii = 0.12 + 0.18 * torch.rand(14, generator=gen)
    colours = torch.rand(14, 3, generator=gen) * 0.8
    dist = (xy[..., None, :] - centres.to(xy)).norm(dim=-1)  # (..., 14)
    inside = dist < radii.to(xy)
    disc = torch.where(inside.any(-1, keepdim=True), colours.to(xy)[inside.float().argmax(-1)], rgb)
    on_poster = ((x.abs() <= POSTER[0]) & (y.abs() <= POSTER[1]))[..., None]
    return torch.where(on_poster, disc, torch.full_like(rgb, BACKGROUND))


def look_at(eye, target, roll: float) -> torch.Tensor:
    """World → camera rotation, rows = (right, down, forward) so image v grows downwards; roll about forward."""
    eye, target = torch.tensor(eye, dtype=torch.float32), torch.tensor(target, dtype=torch.float32)
    forward = (target - eye) / (target - eye).norm()
    right = torch.linalg.cross(forward, torch.tensor([0.0, 1.0, 0.0]))
    right = right / right.norm()
    down = torch.linalg.cross(forward, right)
    c, s = math.cos(roll), math.sin(roll)
    return torch.stack([c * right + s * down, -s * right + c * down, forward])


def plane_homography(eye, target, roll: float, focal: float, size: tuple[int, int]) -> torch.Tensor:
    """H = K [r₁ r₂ t] maps plane points (X, Y, 1) on z = 0 to image points (u, v, 1) up to scale: (3, 3)."""
    R = look_at(eye, target, roll)
    t = -R @ torch.tensor(eye, dtype=torch.float32)
    K = torch.tensor([[focal, 0, (size[0] - 1) / 2], [0, focal, (size[1] - 1) / 2], [0, 0, 1.0]])
    return K @ torch.stack([R[:, 0], R[:, 1], t], dim=1)


CAMERAS = [
    {"eye": (-0.75, -0.25, 3.0), "target": (-0.65, 0.0, 0.0), "roll": 0.0},
    {"eye": (0.9, 0.35, 2.7), "target": (0.7, 0.1, 0.0), "roll": 0.18},
]


def views(size: tuple[int, int], focal: float, device: torch.device):
    """Plane-to-image homographies (2, 3, 3), the rendered views (2, H, W, 3) and the true view-1 → view-2 H."""
    w, h = size
    Hs = torch.stack([plane_homography(c["eye"], c["target"], c["roll"], focal, size) for c in CAMERAS])
    # Render at 2× and average 2 × 2 blocks (anti-aliasing); sub-pixel centres sit at ±1/4 pixel.
    s_u = torch.arange(2 * w) / 2 - 0.25
    s_v = torch.arange(2 * h) / 2 - 0.25
    vv, uu = torch.meshgrid(s_v, s_u, indexing="ij")
    pix = torch.stack([uu, vv, torch.ones_like(uu)], -1)  # (2H, 2W, 3) homogeneous pixel coordinates
    imgs = []
    for H in Hs:
        plane = torch.einsum("ij,hwj->hwi", torch.linalg.inv(H), pix)  # back-project every pixel onto the plane
        img = texture(plane[..., :2] / plane[..., 2:])
        imgs.append(img.reshape(h, 2, w, 2, 3).mean(dim=(1, 3)))
    H12 = Hs[1] @ torch.linalg.inv(Hs[0])
    return Hs.to(device), torch.stack(imgs).to(device), (H12 / H12[2, 2]).to(device)


def matches(Hs: torch.Tensor, size: tuple[int, int], n: int, noise: float, outlier_frac: float, seed: int):
    """Synthetic feature matches: n poster points seen by both cameras, Gaussian pixel noise, and a fraction of the
    view-2 points replaced by random positions (wrong matches). Returns x1, x2 (M, 2) and is_outlier (M,)."""
    gen = torch.Generator().manual_seed(seed)
    dev = Hs.device
    pts = (torch.rand(4 * n, 2, generator=gen) * 2 - 1) * torch.tensor(POSTER) * 0.97
    homog = torch.cat([pts, torch.ones(len(pts), 1)], 1).to(dev)
    proj = torch.einsum("cij,nj->cni", Hs, homog)
    uv = proj[..., :2] / proj[..., 2:]  # (2, 4n, 2)
    bound = torch.tensor([size[0] - 1, size[1] - 1], device=dev, dtype=torch.float32)
    seen = ((uv >= 2) & (uv <= bound - 2)).all(-1).all(0)
    x1, x2 = uv[0, seen][:n], uv[1, seen][:n]
    m = len(x1)
    x1 = x1 + noise * torch.randn(m, 2, generator=gen).to(dev)
    x2 = x2 + noise * torch.randn(m, 2, generator=gen).to(dev)
    is_outlier = (torch.rand(m, generator=gen) < outlier_frac).to(dev)
    random_pts = (torch.rand(m, 2, generator=gen).to(dev) * bound).to(x2)
    x2 = torch.where(is_outlier[:, None], random_pts, x2)
    return x1, x2, is_outlier


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/graphics/homography_stitching/solve.py
"""Robust estimation by batched RANSAC, then warping and feather-blending the two views into one panorama."""

import torch
import torch.nn.functional as F

from .equation import apply_homography, dlt, transfer_error


def ransac(x1: torch.Tensor, x2: torch.Tensor, n_hyp: int, threshold: float, gen: torch.Generator, rounds: int = 2):
    """Fit n_hyp homographies to random 4-match samples at once, keep the one with most inliers, then refit by DLT
    on all its inliers (repeated `rounds` times). x1, x2 (N, 2) → H (3, 3), inliers (N,) bool, counts (n_hyp,)."""
    n = x1.shape[0]
    sample = torch.rand(n_hyp, n, generator=gen).argsort(dim=1)[:, :4].to(x1.device)  # (B, 4) distinct matches
    H_all = dlt(x1[sample], x2[sample])  # (B, 3, 3): one batched SVD for every hypothesis
    err = transfer_error(H_all, x1, x2)  # (B, N)
    counts = (err < threshold).sum(dim=1)  # consensus of each hypothesis
    best = counts.argmax()
    inliers = err[best] < threshold
    H = H_all[best]
    for _ in range(rounds):  # least-squares refit on the consensus set, then re-classify
        H = dlt(x1[inliers][None], x2[inliers][None])[0]
        inliers = transfer_error(H[None], x1, x2)[0] < threshold
    return H, inliers, counts


def sample_image(img: torch.Tensor, uv: torch.Tensor) -> torch.Tensor:
    """Bilinear lookup of img (H, W, 3) at pixel coordinates uv (h, w, 2) → (h, w, 3); zero outside."""
    h, w = img.shape[:2]
    grid = torch.stack([(2 * uv[..., 0] + 1) / w - 1, (2 * uv[..., 1] + 1) / h - 1], -1)  # align_corners=False
    out = F.grid_sample(img.permute(2, 0, 1)[None], grid[None], mode="bilinear", align_corners=False)
    return out[0].permute(1, 2, 0)


def feather(uv: torch.Tensor, size: tuple[int, int]) -> torch.Tensor:
    """Blending weight: distance (in pixels) to the nearest image border, 0 outside the image.  (h, w, 2) → (h, w)"""
    w, h = size
    d = torch.minimum(
        torch.minimum(uv[..., 0] + 0.5, w - 0.5 - uv[..., 0]), torch.minimum(uv[..., 1] + 0.5, h - 0.5 - uv[..., 1])
    )
    return d.clamp(min=0)


def stitch(imgs: torch.Tensor, H12: torch.Tensor, pad_colour: float = 1.0):
    """Panorama in the frame of view 1: every canvas pixel p looks up view 1 at p and view 2 at H12 p, and the
    two are blended with feathering weights. imgs (2, H, W, 3) → panorama (h, w, 3)."""
    h, w = imgs.shape[1:3]
    size = (w, h)
    corners = torch.tensor([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]], dtype=imgs.dtype, device=imgs.device)
    warped = apply_homography(torch.linalg.inv(H12)[None], corners)[0]  # view-2 corners in view-1 coordinates
    allc = torch.cat([corners, warped])
    limit = torch.tensor([2.0 * w, 2.0 * h], device=imgs.device)  # a wild H must not ask for a giant canvas
    lo = torch.maximum(allc.min(0).values.floor(), -limit)
    hi = torch.minimum(allc.max(0).values.ceil(), limit + torch.tensor([w, h], device=imgs.device))
    us = torch.arange(int(lo[0]), int(hi[0]) + 1, device=imgs.device, dtype=imgs.dtype)
    vs = torch.arange(int(lo[1]), int(hi[1]) + 1, device=imgs.device, dtype=imgs.dtype)
    vv, uu = torch.meshgrid(vs, us, indexing="ij")
    p1 = torch.stack([uu, vv], -1)  # (h', w', 2) canvas pixels in view-1 coordinates
    p2 = apply_homography(H12[None], p1.reshape(-1, 2))[0].reshape(p1.shape)
    w1, w2 = feather(p1, size), feather(p2, size)
    blend = (w1[..., None] * sample_image(imgs[0], p1) + w2[..., None] * sample_image(imgs[1], p2)) / (w1 + w2).clamp(
        min=1e-9
    )[..., None]
    covered = (w1 + w2) > 0
    pano = torch.where(covered[..., None], blend, torch.full_like(blend, pad_colour))
    return pano, (int(lo[0]), int(lo[1]))


def corner_error(H: torch.Tensor, H_true: torch.Tensor, size: tuple[int, int]) -> float:
    """Mean distance (pixels) between where H and the true homography send the four image corners."""
    w, h = size
    c = torch.tensor([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]], dtype=H.dtype, device=H.device)
    return float((apply_homography(H[None], c) - apply_homography(H_true[None], c)).norm(dim=-1).mean())


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/graphics/homography_stitching/plot.py
"""The two views with their matches, the RANSAC panorama next to the naive one, and the reprojection errors."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import save_png


def figure(imgs, x1, x2, is_out, inliers, pano_r, pano_a, e_ransac, e_all, threshold, corner_err, path: Path):
    imgs, x1, x2 = imgs.cpu().numpy(), x1.cpu().numpy(), x2.cpu().numpy()
    is_out, inliers = is_out.cpu().numpy(), inliers.cpu().numpy()
    h, w = imgs.shape[1:3]
    gap = 8
    pair = np.ones((h, 2 * w + gap, 3))
    pair[:, :w], pair[:, w + gap :] = imgs[0], imgs[1]

    fig = plt.figure(figsize=(10, 7.4))
    grid = fig.add_gridspec(2, 3, height_ratios=[1, 1.05])
    ax = fig.add_subplot(grid[0, :2])
    ax.imshow(pair, interpolation="nearest")
    for a, b, good in zip(x1, x2, inliers):
        ax.plot(
            [a[0], b[0] + w + gap],
            [a[1], b[1]],
            color="#2563eb" if good else "#dc2626",
            lw=0.6 if good else 0.5,
            alpha=0.75 if good else 0.6,
        )
    ax.scatter(x1[:, 0], x1[:, 1], s=6, c=np.where(inliers, "#1e40af", "#dc2626"), zorder=3)
    ax.scatter(x2[:, 0] + w + gap, x2[:, 1], s=6, c=np.where(inliers, "#1e40af", "#dc2626"), zorder=3)
    ax.set_title(
        f"view 1 | view 2: {inliers.sum()} RANSAC inliers (blue), {(~inliers).sum()} rejected (red)", fontsize=10
    )
    ax.set_axis_off()

    ax_e = fig.add_subplot(grid[0, 2])
    order = np.argsort(e_ransac.cpu().numpy())
    idx = np.arange(len(order))
    er, ea = e_ransac.cpu().numpy()[order], e_all.cpu().numpy()[order]
    ax_e.semilogy(idx, ea, ".", ms=3, color="#9ca3af", label="DLT on all matches")
    ax_e.semilogy(idx[~is_out[order]], er[~is_out[order]], ".", ms=4, color="#2563eb", label="RANSAC: true match")
    ax_e.semilogy(idx[is_out[order]], er[is_out[order]], ".", ms=4, color="#dc2626", label="RANSAC: wrong match")
    ax_e.axhline(threshold, color="#f59e0b", lw=1.2, ls="--", label=f"threshold {threshold:g} px")
    ax_e.set_xlabel("match (sorted by RANSAC error)")
    ax_e.set_ylabel("reprojection error |x₂ − H x₁| (px)")
    ax_e.set_title("reprojection error", fontsize=10)
    ax_e.legend(fontsize=7, loc="upper left")

    for k, (pano, label) in enumerate(
        [
            (pano_r, f"panorama, RANSAC + DLT (corners off by {corner_err[0]:.1f} px)"),
            (pano_a, f"panorama, DLT on all matches ({corner_err[1]:.0f} px)"),
        ]
    ):
        axp = fig.add_subplot(grid[1, 0:2] if k == 0 else grid[1, 2])
        axp.imshow(pano.clamp(0, 1).cpu().numpy(), interpolation="nearest")
        axp.set_title(label, fontsize=10)
        axp.set_axis_off()
    fig.tight_layout()
    return save_png(fig, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'matches': 120,
    'outliers': 0.35,
    'noise': 0.7,
    'hypotheses': 4000,
    'threshold': 3.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.graphics.homography_stitching")]:
    del sys.modules[name]
main = importlib.import_module("problems.graphics.homography_stitching.main")
problem_dir = Path("problems/graphics/homography_stitching")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
